# 02 Baselines

- **Model 0:** majority class, the floor every model must beat.
- **Model 1:** logistic regression on TF-IDF character 1-3-grams of the domain, trained with the sample weights.

Data: the saved splits in `data/splits/` (made by `00_data_splits.ipynb`, checked against `manifest.json`).

Each is run in two setups: **binary** (target DGA/legitimate) and **family-first** (target `legit` or a DGA family, collapsed with P(DGA) = 1 - P(legit)), with and without the TLD (removed with `tldextract`, so multi-part suffixes such as `co.uk` are removed whole). Metrics are computed on the DGA/legitimate decision, on the validation set; the test set is kept for the final results.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Settings

`SAMPLE` trains on a random subset of the training set for quick runs; set it to `None` to use all 1.6M rows (the family setup then takes much longer).

In [2]:
SAMPLE = 100_000   # None = full training set
SEED = 0
EVAL_SET = "val"   # "test" only for the final results

In [3]:
import time

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from src.data.dataset import load_splits

train, val, test = load_splits()
if SAMPLE:
    train = train.sample(n=SAMPLE, random_state=SEED)
evaluate = test if EVAL_SET == "test" else val
y_eval = (evaluate["label"] == "dga").astype(int).to_numpy()
print(f"train: {len(train):,} rows, {EVAL_SET}: {len(evaluate):,} rows")

train: 100,000 rows, val: 199,936 rows


## Metrics

In [4]:
def binary_metrics(y_true, p_dga, threshold=0.5):
    y_pred = (p_dga >= threshold).astype(int)
    auc = roc_auc_score(y_true, p_dga) if len(np.unique(p_dga)) > 1 else 0.5
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "dga_precision": precision_score(y_true, y_pred, zero_division=0),
        "dga_recall": recall_score(y_true, y_pred),
        "roc_auc": auc,
    }


def recall_per_family(families, y_true, y_pred):
    dga = y_true == 1
    hit = pd.Series(y_pred[dga] == 1, index=families.to_numpy()[dga])
    return hit.groupby(level=0).mean().sort_values()

## Model 0: majority class

In [5]:
majority = train["label"].value_counts().idxmax()
p_majority = np.full(len(evaluate), float(majority == "dga"))
results = {("Model 0", "-", "yes"): binary_metrics(y_eval, p_majority)}
print("majority class:", majority)

majority class: dga


## Model 1: logistic regression

In [6]:
import tldextract

# The suffix list bundled with tldextract (no download), so every machine splits
# domains the same way.
extract = tldextract.TLDExtract(suffix_list_urls=())


def strip_tld(domains):
    """Remove the public suffix: google.com -> google, bbc.co.uk -> bbc."""
    def name(domain):
        parts = extract(domain)
        if not parts.suffix:
            # TLDs outside the public suffix list (.bazar, .bit: BazarBackdoor and
            # Necurs use alternative DNS roots): the last label is the TLD.
            return domain.rsplit(".", 1)[0]
        return ".".join(p for p in (parts.subdomain, parts.domain) if p)
    return domains.map(name)


def run_logreg(setup, tld=True):
    x_train, x_eval = train["domain"], evaluate["domain"]
    if not tld:
        x_train, x_eval = strip_tld(x_train), strip_tld(x_eval)
    target = train["label"] if setup == "binary" else train["family"]
    vec = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=5, sublinear_tf=True)
    clf = LogisticRegression(max_iter=300, random_state=SEED)
    clf.fit(vec.fit_transform(x_train), target, sample_weight=train["weight"])
    proba = clf.predict_proba(vec.transform(x_eval))
    return 1 - proba[:, list(clf.classes_).index("legit")]   # P(DGA)


p_dga = {}
for setup in ["binary", "family"]:
    for tld in [True, False]:
        start = time.time()
        p_dga[setup, tld] = run_logreg(setup, tld)
        results["Model 1", setup, "yes" if tld else "no"] = binary_metrics(y_eval, p_dga[setup, tld])
        print(f"{setup:6s} TLD={tld!s:5s} {time.time() - start:6.0f} s")

binary TLD=True       7 s


binary TLD=False      8 s


family TLD=True     238 s


family TLD=False    139 s


## Results (DGA/legitimate decision)

In [7]:
table = pd.DataFrame(results).T.round(4)
table.index.names = ["model", "setup", "TLD"]
table

macro_f1  dga_precision  dga_recall  roc_auc
model   setup  TLD                                              
Model 0 -      yes    0.3333         0.5000      1.0000   0.5000
Model 1 binary yes    0.9333         0.9417      0.9238   0.9804
               no     0.9156         0.9252      0.9042   0.9683
        family yes    0.9424         0.9512      0.9327   0.9852
               no     0.9225         0.9335      0.9098   0.9736

## Recall per DGA family

The 10 families Model 1 misses most (family-first setup, with TLD).

In [8]:
y_pred = (p_dga["family", True] >= 0.5).astype(int)
recall_per_family(evaluate["family"], y_eval, y_pred).head(10).round(3)

matsnu                 0.000
nymaim2                0.113
bigviktor_subdomain    0.128
bigviktor              0.255
virut                  0.458
gozi                   0.468
vawtrak                0.574
qsnatch                0.623
recjs_subdomain        0.667
suppobox               0.702
dtype: float64